# Load the three datasets and make the splits
1. Download HelpSteer2, PKU-SafeRLHF and UltraFeedback in the same format (`prompt`, `chosen`, `rejected`).
2. Split each one into train / val / test and save them to `data/`.

No GPU needed. **Step 2 only needs to run once.** After the files are pushed to GitHub, everyone uses the same splits.

In [18]:
!pip install -r requirements.txt

In [21]:
import os, sys, getpass

if os.path.exists("/content"):   
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    REPO = "/content/mfr-dpo"
else:                            
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")   
import mfr_data

## 1. Load

In [22]:
# optional 
os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF token: ")

In [20]:
datasets = {
    "helpful": mfr_data.load_helpsteer2(),
    "safe": mfr_data.load_pku_saferlhf(),
    "quality": mfr_data.load_ultrafeedback(),
}

In [23]:
import pandas as pd

pd.DataFrame({
    name: {
        "pairs": len(df),
        "unique prompts": df["prompt"].nunique(),
        "avg prompt chars": int(df["prompt"].str.len().mean()),
        "avg chosen chars": int(df["chosen"].str.len().mean()),
        "avg rejected chars": int(df["rejected"].str.len().mean()),
    }
    for name, df in datasets.items()
})

,helpful,safe,quality
pairs,2765,10796,42182
unique prompts,2762,8555,42174
avg prompt chars,390,129,671
avg chosen chars,1463,455,1239
avg rejected chars,1348,528,1003


Do the chosen responses actually look better? Change `i` to see other examples.

In [24]:
i = 0
for name, df in datasets.items():
    row = df.iloc[i]
    print(f"=============== {name} ===============")
    print("PROMPT:  ", row["prompt"][:500])
    print("\nCHOSEN:  ", row["chosen"][:500])
    print("\nREJECTED:", row["rejected"][:500], "\n")

=============== helpful ===============
PROMPT:   Please make a list of independent Fertility coaching and consulting services in USA

CHOSEN:   Sure, here is a list of independent Fertility coaching and consulting services in USA:

1. Fertility Focus LLC
2. Fertility Journey Inc.
3. Fertility Road LLC
4. Fertility Wellness LLC
5. The Fertility Coach LLC
6. Fertility Consulting Services LLC
7. Fertility Health Services LLC
8. Fertility Support Services LLC
9. Fertility Advocates LLC
10. Fertility Resource Group LLC
11. Fertility Solutions LLC
12. Fertility Consulting LLC
13. Fertility Advocates and Solutions LLC
14. Fertility Advocates a

REJECTED: Sure, here are some independent Fertility coaching and consulting services in the USA:

1. Fertility Authority
2. Fertility Solutions
3. Fertility Success Coaching
4. Fertility Consulting Services
5. Fertility Coaching and Consulting
6. Fertility Coaching and Consulting Services
7. Fertility Coaching and Consulting Services
8. Fertility Coac

## 2. Split (run once)
Same sizes for all three datasets, so each training stage gets the same amount of data.
HelpSteer2 is the smallest (about 2,765 pairs before the length limit), which is why train is 2,000.

In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")   # only the tokenizer, not the model
SIZES = {"train": 2000, "val": 200, "test": 300}

splits = mfr_data.make_splits(datasets, tokenizer, SIZES, max_tokens=768, seed=0)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

AttributeError: module 'mfr_data' has no attribute 'make_splits'

In [ ]:
# token lengths in the train splits
pd.concat({name: parts["train"][["prompt_tokens", "chosen_tokens", "rejected_tokens"]].describe().round()
           for name, parts in splits.items()}, axis=1)

In [ ]:
DATA_DIR = f"{REPO}/data"
mfr_data.save_splits(splits, DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

Then commit the `data/` folder to GitHub.

* **Ran this on your laptop:** `git add data && git commit -m "Add splits" && git push`
* **Ran this on Colab:** the files are on the Colab machine. Download them from the Files panel, put them in your local `data/` folder, and push.

Anyone can load them later with:
```python
splits = mfr_data.load_splits(f"{REPO}/data")
splits["helpful"]["train"]
```